In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

In [ ]:
from demand_forecasting.data import read_source_data
df = read_source_data('Data/data.csv')
df["Date"] = pd.to_datetime(df["Date"])
df.head()

In [ ]:
from demand_forecasting.data import read_source_data

df = read_source_data('Data/data.csv')
df["Date"] = pd.to_datetime(df["Date"])

#We remove the nans for the target value
df = df.dropna(subset=["Consumption (MW)"]).copy()

data = (
    df.groupby("Date")["Consumption (MW)"]
      .agg(list)
      .reset_index(name="Consumption")
)

#The dataframe is sorted to avoid training on future data
data = data.sort_values(["Date"])



consumption = data["Consumption"]
#lag 1 is the data from yesterday, on which we do the regression to predict today's data
data["lag_1"] = consumption.shift(1)
data= data.iloc[1:]
data= data.drop(["Date"], axis=1)

n = len(data)
cut_train = int(0.70 * n)
cut_val = int(0.85 * n)

data_train = data.iloc[:cut_train].copy()
data_val = data.iloc[cut_train:cut_val].copy()
data_test = data.iloc[cut_val:].copy()

target = "Consumption"

X_train = data_train.drop(columns=[target])
y_train = data_train[target]

X_test = data_test.drop(columns=[target])
y_test = data_test[target]

X_val= data_val.drop(columns=[target])
y_val= data_val[target]

print(X_val)

In [ ]:
data= data.iloc[1:]
data.head()

In [ ]:
print(df.isna().mean().sort_values(ascending=False))

In [ ]:
df = df.dropna(subset=["Consumption (MW)"]).copy()

In [ ]:
raw_dates = data["Timestamp"].copy()

data["Timestamp"] = pd.to_datetime(
    raw_dates,
    utc=True,
    errors="coerce"
)

print(data["Timestamp"].head())
print(data["Timestamp"].dtype)

In [ ]:
failures = raw_dates.notna() & df["Timestamp"].isna()

print("Unparsed dates :", failures.sum())
print(raw_dates[failures].head())

In [ ]:
local_dates = data["Timestamp"].dt.tz_convert("Europe/Paris")

data["hour"] = (
    local_dates.dt.hour
    + local_dates.dt.minute / 60
)
data["weekday"] = local_dates.dt.dayofweek  # Monday=0, Sunday=6
data["month"] = local_dates.dt.month

In [ ]:
data['weekday'].unique()

In [ ]:
plt.plot([data.loc[data["weekday"] == i, "Consumption (MW)"].mean() for i in range(0,7)])

In [ ]:
data = data.sort_values("Timestamp").reset_index(drop=True)

# Recreate this variable after sorting
consumption = data["Consumption (MW)"]

data["lag_1"] = consumption.shift(1)
data["lag_48"] = consumption.shift(48)
data["lag_336"] = consumption.shift(336)
data["rolling_mean_24h"] = consumption.shift(1).rolling(48).mean()

In [ ]:
display(
    data.iloc[336:341][
        ["Timestamp", "Consumption (MW)",
         "lag_1", "lag_48", "lag_336", "rolling_mean_24h"]
    ]
)

In [ ]:
data= data.drop(["Timestamp"], axis=1).iloc[336:]
shape= data.shape[0]


In [ ]:
n = len(data)
cut_train = int(0.70 * n)
cut_val = int(0.85 * n)

data_train = data.iloc[:cut_train].copy()
data_val = data.iloc[cut_train:cut_val].copy()
data_test = data.iloc[cut_val:].copy()

In [ ]:
target = "Consumption (MW)"

X_train = data_train.drop(columns=[target])
y_train = data_train[target]

X_test = data_test.drop(columns=[target])
y_test = data_test[target]

X_val= data_val.drop(columns=[target])
y_val= data_val[target]

In [ ]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()

X_train = scaler.fit_transform(X_train)
X_val = scaler.transform(X_val)
X_test = scaler.transform(X_test)

In [ ]:
X_train